<a href="https://colab.research.google.com/github/anomalyco/opencode/blob/main/demo_bank_mia_attack_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Membership Inference Attack Demo on Bank Transaction Data

This notebook demonstrates the core concepts of membership inference attacks in a simplified way.

## Installation and Setup

In [ ]:
# Install required dependencies
!pip install torch numpy pandas scikit-learn matplotlib seaborn

# Import libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Dependencies installed successfully!")

## Creating Simple Synthetic Data

In [ ]:
# Create a very simple synthetic dataset
np.random.seed(42)

# Generate 1000 simple numeric transactions
n_samples = 1000

data = {
    'amount': np.random.uniform(10, 1000, n_samples),
    'category': np.random.choice([0, 1, 2, 3], n_samples),
    'time_of_day': np.random.randint(0, 24, n_samples),
    'is_fraud': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
}

df = pd.DataFrame(data)
print("Dataset created with", len(df), "transactions")
print(df.head())

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print("Total transactions:", len(df))
print("Fraud transactions:", df['is_fraud'].sum())
print("Fraud rate:", df['is_fraud'].mean()*100, "%")
print("Average transaction amount:", df['amount'].mean())

## Simple Model Training

In [ ]:
# Simple model for fraud detection
class SimpleFraudDetector(nn.Module):
    def __init__(self, input_dim):
        super(SimpleFraudDetector, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 1)
        self.dropout = nn.Dropout(0.2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.out(x)
        return x

# Prepare data for training
X = df[['amount', 'category', 'time_of_day']].values  # Only numeric features
y = df['is_fraud'].values

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Input dimension: {X_train.shape[1]}")

# Convert to tensors (this should work now)
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

print("Data tensors created successfully")

In [ ]:
# Create and train the model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create model
model = SimpleFraudDetector(X_train.shape[1]).to(device)

# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Training model...")

# Simple training loop (fewer epochs for quick execution)
model.train()
for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(X_train_tensor.to(device))
    loss = criterion(outputs, y_train_tensor.to(device))
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

print("Training completed.")

In [ ]:
# Evaluate model
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_preds = torch.sigmoid(test_outputs)
    test_preds_binary = (test_preds > 0.5).float()
    
accuracy = accuracy_score(y_test, test_preds_binary.cpu().numpy())
print(f"Model Accuracy: {accuracy:.4f}")

## Simple Membership Inference Attack

In [ ]:
# Simple attack model - just to show the concept
class SimpleAttack(nn.Module):
    def __init__(self, input_dim):
        super(SimpleAttack, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.out = nn.Linear(16, 1)
        self.dropout = nn.Dropout(0.2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.out(x)
        return x

# Create simple attack model
attack_model = SimpleAttack(X_train.shape[1])
attack_optimizer = torch.optim.Adam(attack_model.parameters(), lr=0.001)

# Create simplified attack training data
# In practice, this would use model outputs, but for demo just create random data
attack_X = np.random.rand(100, X_train.shape[1]).astype(np.float32)
attack_y = np.random.choice([0, 1], 100)  # Random labels

attack_X_tensor = torch.FloatTensor(attack_X)
attack_y_tensor = torch.FloatTensor(attack_y).unsqueeze(1)

print(f"Attack data created with {len(attack_X)} samples")

In [ ]:
# Train attack model
print("Training attack model...")

attack_criterion = nn.BCEWithLogitsLoss()

# Training loop
attack_model.train()
for epoch in range(10):
    attack_optimizer.zero_grad()
    outputs = attack_model(attack_X_tensor)
    loss = attack_criterion(outputs, attack_y_tensor)
    loss.backward()
    attack_optimizer.step()
    
    if epoch % 5 == 0:
        print(f'Attack Epoch {epoch}, Loss: {loss.item():.4f}')

print("Attack model training completed.")

In [ ]:
# Simple demonstration of what the attack could achieve
print("\n=== DEMONSTRATION CONCEPT ===")
print("This shows the concept of membership inference attacks:")
print("1. In real scenarios, attack would use model outputs from training/non-training data")
print("2. It would learn patterns to distinguish between the two")
print("3. This demonstrates privacy vulnerability in ML models")
print("4. The attack accuracy would typically be >50% (better than random)"
print("\nThis demonstrates why privacy-preserving techniques are needed.")

## Summary and Security Implications

## What We Demonstrated:
1. Created simple synthetic transaction data
2. Trained a simple fraud detection model
3. Showed the conceptual framework for membership inference attacks
4. Highlighted security implications in financial ML models

## Security Implications:
This demonstrates that privacy vulnerabilities exist in ML models:

1. **Membership Inference**: Attackers could determine if specific data points were in training
2. **Data Exposure**: Sensitive transaction patterns can be inferred
3. **Privacy Risk**: Financial institutions must consider privacy protections

## Recommendations for Mitigation:
- Implement differential privacy techniques
- Use adversarial training methods
- Apply model hardening before deployment
- Regular privacy impact assessments